<a href="https://colab.research.google.com/github/Saikadam123/ADM-Project/blob/main/Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Ground Truth Binary

In [ ]:
# Creates binary ground-truth labels directly from the Problems field.
# A report is labeled 1 if any predefined abnormality is present and 0 otherwise.

import os
import re
import pandas as pd

REPORTS_PATH = "/content/drive/MyDrive/Dataset/indiana_reports.csv"

OUTPUT_PATH = (
    "/content/drive/MyDrive/Dataset/"
    "ground_truth_binary.csv"
)

ABNORMAL_TERMS = [
    "atelectasis",
    "cardiomegaly",
    "cardiac enlargement",
    "pleural effusion",
    "effusion",
    "infiltrate",
    "infiltration",
    "mass",
    "nodule",
    "pneumonia",
    "pneumothorax",
    "edema",
    "pulmonary edema",
    "emphysema",
    "fibrosis",
    "pleural thickening",
    "hernia"
]

ABNORMAL_PATTERN = (
    r"(?:"
    + "|".join(
        re.escape(term)
        for term in ABNORMAL_TERMS
    )
    + r")"
)

df = pd.read_csv(REPORTS_PATH)

# Keep reports with available findings
df = df[
    df["findings"].notna()
    & (df["findings"].str.strip() != "")
].reset_index(drop=True)

# 1 = any recognized abnormality
# 0 = no recognized abnormality
df["binary_label"] = (
    df["Problems"]
    .fillna("")
    .astype(str)
    .str.contains(
        ABNORMAL_PATTERN,
        case=False,
        regex=True,
        na=False
    )
    .astype(int)
)

binary_gt = df[
    ["uid", "binary_label"]
].copy()

# Basic validation
assert binary_gt["binary_label"].isin([0, 1]).all()
assert binary_gt["uid"].duplicated().sum() == 0

os.makedirs(
    os.path.dirname(OUTPUT_PATH),
    exist_ok=True
)

binary_gt.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Binary ground truth saved:", OUTPUT_PATH)

print("\nClass distribution:")
print(
    binary_gt["binary_label"]
    .value_counts()
    .sort_index()
)

Binary ground truth saved: /content/drive/MyDrive/Dataset/ground_truth_binary.csv

Class distribution:
binary_label
0    2538
1     799
Name: count, dtype: int64


#Build Multimodal Dataset Before Embedding

In [ ]:
import pandas as pd

In [ ]:
# Cleans and aligns Indiana radiology reports, frontal chest X-rays, and binary diagnostic labels by UID into a unified multimodal dataset and saves it to CSV.


REPORTS_PATH = "/content/drive/MyDrive/Dataset/indiana_reports.csv"
IMAGES_PATH = "/content/drive/MyDrive/Dataset/indiana_projections.csv"
LABELS_PATH = "/content/drive/MyDrive/Dataset/ground_truth_binary.csv"
OUTPUT_PATH = "/content/drive/MyDrive/Dataset/multimodal_master_binary_dataset.csv"

reports_df = pd.read_csv(REPORTS_PATH)
images_df = pd.read_csv(IMAGES_PATH)
labels_df = pd.read_csv(LABELS_PATH)

text_df = reports_df[["uid", "findings"]].copy()
text_df = text_df.dropna(subset=["findings"])
text_df = text_df[text_df["findings"].astype(str).str.strip() != ""].copy()
text_df["findings"] = text_df["findings"].astype(str).str.strip()
text_df = text_df.drop_duplicates(subset="uid", keep="first")

images_df = images_df.dropna(subset=["projection"])
frontal_images_df = images_df[
    images_df["projection"].astype(str).str.strip().str.lower() == "frontal"
][["uid", "filename"]].copy()
frontal_images_df = frontal_images_df.drop_duplicates(subset="uid", keep="first")

binary_labels_df = labels_df.drop_duplicates(
    subset="uid",
    keep="first"
).copy()

assert binary_labels_df["binary_label"].isin([0, 1]).all()
assert binary_labels_df["uid"].duplicated().sum() == 0

master_df = pd.merge(
    text_df, frontal_images_df, on="uid", how="inner", validate="one_to_one"
)
master_df = pd.merge(
    master_df, binary_labels_df, on="uid", how="inner", validate="one_to_one"
)

master_df.to_csv(OUTPUT_PATH, index=False)

# Stratification & Spliting

In [ ]:
# Splits the binary multimodal dataset into stratified training (70%), validation (15%), and test (15%) subsets and saves them as CSV files.
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

master_df = pd.read_csv(
    "/content/drive/MyDrive/Dataset/multimodal_master_binary_dataset.csv"
)

X = master_df[["uid", "findings", "filename"]].copy()
y = master_df["binary_label"].values

sss_1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(sss_1.split(X, y))

train_df = master_df.iloc[train_idx].reset_index(drop=True)
temp_df = master_df.iloc[temp_idx].reset_index(drop=True)

X_temp = temp_df[["uid", "findings", "filename"]]
y_temp = temp_df["binary_label"].values

sss_2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(sss_2.split(X_temp, y_temp))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

OUTPUT_DIR = "/content/drive/MyDrive/Dataset/"

train_df.to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
val_df.to_csv(f"{OUTPUT_DIR}/validation.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test.csv", index=False)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("\nTraining class distribution:")
print(train_df["binary_label"].value_counts(normalize=True).sort_index())

print("\nValidation class distribution:")
print(val_df["binary_label"].value_counts(normalize=True).sort_index())

print("\nTest class distribution:")
print(test_df["binary_label"].value_counts(normalize=True).sort_index())

# Image Embedding

In [ ]:
# train, validation, test image embedded

In [ ]:
# Loads the pretrained Swin Transformer and image processor and configures
# them for fixed image feature extraction across train, validation, and test sets.

import os
import torch
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel


IMAGE_FOLDER = "/content/drive/MyDrive/Dataset/images_normalized"
OUTPUT_FOLDER = "/content/drive/MyDrive/Dataset/image_embeddings"

IMAGE_MODEL_NAME = "microsoft/swin-tiny-patch4-window7-224"

BATCH_SIZE = 16

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

image_processor = AutoImageProcessor.from_pretrained(
    IMAGE_MODEL_NAME
)

image_encoder = AutoModel.from_pretrained(
    IMAGE_MODEL_NAME
).to(device)

image_encoder.eval()

print("Using device:", device)
print("Model:", IMAGE_MODEL_NAME)
print("Batch size:", BATCH_SIZE)

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

[transformers] SwinModel LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda
Model: microsoft/swin-tiny-patch4-window7-224
Batch size: 16


In [ ]:
# Checks that all image files referenced in each data split are available
# before starting embedding extraction.

def check_images(df, split_name):

    missing_images = []

    for filename in df["filename"]:

        image_path = os.path.join(
            IMAGE_FOLDER,
            filename
        )

        if not os.path.exists(image_path):
            missing_images.append(filename)

    print(
        f"{split_name}: "
        f"{len(df)} samples, "
        f"{len(missing_images)} missing images"
    )

    if missing_images:
        print(
            "First missing images:",
            missing_images[:10]
        )

    return missing_images


train_missing = check_images(
    train_df,
    "Training"
)

val_missing = check_images(
    val_df,
    "Validation"
)

test_missing = check_images(
    test_df,
    "Test"
)

Training: 2239 samples, 0 missing images
Validation: 480 samples, 0 missing images
Test: 480 samples, 0 missing images


In [ ]:
# Extracts 768-dimensional image embeddings using the pretrained Swin encoder.
# Images are processed in batches, converted to RGB, preprocessed with the
# Swin image processor, and represented by the pooled Swin feature vector.

def extract_image_embeddings(df, split_name):

    all_embeddings = []
    all_uids = []
    all_filenames = []

    with torch.no_grad():

        for start in tqdm(
            range(0, len(df), BATCH_SIZE),
            desc=f"Extracting {split_name} image embeddings"
        ):

            batch_df = df.iloc[
                start:start + BATCH_SIZE
            ]

            batch_images = []
            batch_uids = []
            batch_filenames = []

            # Load images in the current batch
            for _, row in batch_df.iterrows():

                filename = row["filename"]

                image_path = os.path.join(
                    IMAGE_FOLDER,
                    filename
                )

                with Image.open(image_path) as img:
                    image = img.convert("RGB")

                batch_images.append(image)
                batch_uids.append(row["uid"])
                batch_filenames.append(filename)

            # Preprocess images for Swin
            image_inputs = image_processor(
                images=batch_images,
                return_tensors="pt"
            )

            pixel_values = (
                image_inputs["pixel_values"]
                .to(device)
            )

            # Pass images through the pretrained Swin encoder
            outputs = image_encoder(
                pixel_values=pixel_values
            )

            # One 768-dimensional pooled representation per image
            embeddings = (
                outputs.pooler_output
                .cpu()
            )

            all_embeddings.append(embeddings)
            all_uids.extend(batch_uids)
            all_filenames.extend(batch_filenames)


    image_embeddings = torch.cat(
        all_embeddings,
        dim=0
    )

    # Verify extraction
    assert image_embeddings.shape[0] == len(df)
    assert image_embeddings.shape[1] == 768
    assert len(all_uids) == len(df)
    assert len(all_filenames) == len(df)

    print(
        f"{split_name} image embeddings shape:",
        image_embeddings.shape
    )

    return (
        image_embeddings,
        all_uids,
        all_filenames
    )

In [ ]:
# Applies the same Swin embedding procedure independently to
# the training, validation, and test image sets.

train_image_embeddings, train_image_uids, train_image_filenames = (
    extract_image_embeddings(
        train_df,
        "Training"
    )
)

val_image_embeddings, val_image_uids, val_image_filenames = (
    extract_image_embeddings(
        val_df,
        "Validation"
    )
)

test_image_embeddings, test_image_uids, test_image_filenames = (
    extract_image_embeddings(
        test_df,
        "Test"
    )
)

Extracting Training image embeddings:   0%|          | 0/140 [00:00<?, ?it/s]

Training image embeddings shape: torch.Size([2239, 768])


Extracting Validation image embeddings:   0%|          | 0/30 [00:00<?, ?it/s]

Validation image embeddings shape: torch.Size([480, 768])


Extracting Test image embeddings:   0%|          | 0/30 [00:00<?, ?it/s]

Test image embeddings shape: torch.Size([480, 768])


In [ ]:
# Stores each image embedding with its corresponding UID and filename
# in a CSV file for later multimodal feature combination.

def save_image_embeddings(
    embeddings,
    uids,
    filenames,
    output_path
):

    embedding_vectors = (
        embeddings
        .cpu()
        .numpy()
        .tolist()
    )

    embedding_df = pd.DataFrame({
    "uid": uids,
    "filename": filenames,
    "feature_vector": [
        json.dumps(vector)
        for vector in embedding_vectors
    ]
})

    embedding_df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {len(embedding_df)} embeddings to:",
        output_path
    )

In [ ]:
# Saves the image embeddings for all three data splits as separate CSV files.

TRAIN_IMAGE_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "train_image_embeddings_vectors.csv"
)

VAL_IMAGE_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "val_image_embeddings_vectors.csv"
)

TEST_IMAGE_OUTPUT = os.path.join(
    OUTPUT_FOLDER,
    "test_image_embeddings_vectors.csv"
)


save_image_embeddings(
    train_image_embeddings,
    train_image_uids,
    train_image_filenames,
    TRAIN_IMAGE_OUTPUT
)

save_image_embeddings(
    val_image_embeddings,
    val_image_uids,
    val_image_filenames,
    VAL_IMAGE_OUTPUT
)

save_image_embeddings(
    test_image_embeddings,
    test_image_uids,
    test_image_filenames,
    TEST_IMAGE_OUTPUT
)

Saved 2239 embeddings to: /content/drive/MyDrive/Dataset/image_embeddings/train_image_embeddings_vectors.csv
Saved 480 embeddings to: /content/drive/MyDrive/Dataset/image_embeddings/val_image_embeddings_vectors.csv
Saved 480 embeddings to: /content/drive/MyDrive/Dataset/image_embeddings/test_image_embeddings_vectors.csv


# Text Embedding

In [ ]:
# train, validation, test text embedded

In [ ]:
# Loads the pretrained BioClinicalBERT tokenizer and encoder for
# fixed text feature extraction across train, validation, and test sets.

import os
import json
import torch
import pandas as pd

from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


TEXT_OUTPUT_FOLDER = "/content/drive/MyDrive/Dataset/text_embeddings"

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

MAX_LEN = 256
BATCH_SIZE = 32

os.makedirs(
    TEXT_OUTPUT_FOLDER,
    exist_ok=True
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

text_encoder = AutoModel.from_pretrained(
    MODEL_NAME
).to(device)

text_encoder.eval()

print("Using device:", device)
print("Model:", MODEL_NAME)
print("Batch size:", BATCH_SIZE)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Using device: cuda
Model: emilyalsentzer/Bio_ClinicalBERT
Batch size: 32


In [ ]:
# Organizes UIDs and radiology findings for batched text embedding extraction.

class FindingsDataset(Dataset):

    def __init__(self, df):
        self.uids = df["uid"].tolist()

        self.findings = (
            df["findings"]
            .astype(str)
            .tolist()
        )

    def __len__(self):
        return len(self.findings)

    def __getitem__(self, idx):
        return (
            self.uids[idx],
            self.findings[idx]
        )


def collate_fn(batch):

    uids, texts = zip(*batch)

    return (
        list(uids),
        list(texts)
    )

In [ ]:
# Tokenizes radiology findings and extracts one 768-dimensional [CLS]
# embedding per report using the pretrained BioClinicalBERT encoder.

def extract_text_embeddings(
    df,
    split_name
):

    loader = DataLoader(
        FindingsDataset(df),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    all_uids = []
    all_findings = []
    all_embeddings = []

    with torch.inference_mode():

        for uids, texts in tqdm(
            loader,
            desc=f"Extracting {split_name} text embeddings"
        ):

            # Tokenize, pad, and truncate reports
            encoded = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            ).to(device)

            # Pass tokenized reports through BioClinicalBERT
            outputs = text_encoder(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"]
            )

            # Extract one 768-dimensional [CLS] representation per report
            embeddings = (
                outputs.last_hidden_state[:, 0, :]
                .cpu()
            )

            all_embeddings.append(
                embeddings
            )

            all_uids.extend(
                uids
            )

            all_findings.extend(
                texts
            )


    text_embeddings = torch.cat(
        all_embeddings,
        dim=0
    )

    # Verify extraction
    assert text_embeddings.shape[0] == len(df)
    assert text_embeddings.shape[1] == 768
    assert len(all_uids) == len(df)
    assert len(all_findings) == len(df)

    print(
        f"{split_name} text embeddings shape:",
        text_embeddings.shape
    )

    return (
        text_embeddings,
        all_uids,
        all_findings
    )

In [ ]:
# Applies the same BioClinicalBERT embedding procedure independently
# to the training, validation, and test radiology findings.

train_text_embeddings, train_text_uids, train_findings = (
    extract_text_embeddings(
        train_df,
        "Training"
    )
)

val_text_embeddings, val_text_uids, val_findings = (
    extract_text_embeddings(
        val_df,
        "Validation"
    )
)

test_text_embeddings, test_text_uids, test_findings = (
    extract_text_embeddings(
        test_df,
        "Test"
    )
)

Extracting Training text embeddings:   0%|          | 0/70 [00:00<?, ?it/s]

Training text embeddings shape: torch.Size([2239, 768])


Extracting Validation text embeddings:   0%|          | 0/15 [00:00<?, ?it/s]

Validation text embeddings shape: torch.Size([480, 768])


Extracting Test text embeddings:   0%|          | 0/15 [00:00<?, ?it/s]

Test text embeddings shape: torch.Size([480, 768])


In [ ]:
# Stores each text embedding with its corresponding UID and radiology findings
# in a CSV file for later multimodal feature combination.

def save_text_embeddings(
    embeddings,
    uids,
    findings,
    output_path
):

    embedding_vectors = (
        embeddings
        .numpy()
        .tolist()
    )

    text_embedding_df = pd.DataFrame({
        "uid": uids,
        "findings": findings,
        "feature_vector": [
            json.dumps(vector)
            for vector in embedding_vectors
        ]
    })

    text_embedding_df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved {len(text_embedding_df)} embeddings to:",
        output_path
    )

In [ ]:
# Saves the text embeddings for all three data splits as separate CSV files.

TRAIN_TEXT_OUTPUT = os.path.join(
    TEXT_OUTPUT_FOLDER,
    "train_text_embeddings.csv"
)

VAL_TEXT_OUTPUT = os.path.join(
    TEXT_OUTPUT_FOLDER,
    "val_text_embeddings.csv"
)

TEST_TEXT_OUTPUT = os.path.join(
    TEXT_OUTPUT_FOLDER,
    "test_text_embeddings.csv"
)


save_text_embeddings(
    train_text_embeddings,
    train_text_uids,
    train_findings,
    TRAIN_TEXT_OUTPUT
)

save_text_embeddings(
    val_text_embeddings,
    val_text_uids,
    val_findings,
    VAL_TEXT_OUTPUT
)

save_text_embeddings(
    test_text_embeddings,
    test_text_uids,
    test_findings,
    TEST_TEXT_OUTPUT
)

Saved 2239 embeddings to: /content/drive/MyDrive/Dataset/text_embeddings/train_text_embeddings.csv
Saved 480 embeddings to: /content/drive/MyDrive/Dataset/text_embeddings/val_text_embeddings.csv
Saved 480 embeddings to: /content/drive/MyDrive/Dataset/text_embeddings/test_text_embeddings.csv


# Combined Image and Text Embeddings

In [ ]:
# combine image and text embeddings

In [ ]:
# Loads the saved image, text, and binary ground-truth files and defines
# a helper function for converting stored feature vectors back to arrays.

import os
import json
import numpy as np
import pandas as pd


IMAGE_EMBEDDING_FOLDER = (
    "/content/drive/MyDrive/Dataset/image_embeddings"
)

TEXT_EMBEDDING_FOLDER = (
    "/content/drive/MyDrive/Dataset/text_embeddings"
)

GT_PATH = (
    "/content/drive/MyDrive/Dataset/"
    "ground_truth_binary.csv"
)

COMBINED_OUTPUT_FOLDER = (
    "/content/drive/MyDrive/Dataset"
)

os.makedirs(
    COMBINED_OUTPUT_FOLDER,
    exist_ok=True
)


def parse_vector(value):

    if isinstance(value, str):
        value = json.loads(value)

    return np.asarray(
        value,
        dtype=np.float32
    )

In [ ]:
# Matches text and image embeddings by UID, concatenates their 768-dimensional
# representations into a 1536-dimensional multimodal vector, and attaches
# the corresponding binary label.

def combine_embeddings(
    text_path,
    image_path,
    gt_path,
    output_path,
    split_name
):

    text_df = pd.read_csv(
        text_path
    )

    image_df = pd.read_csv(
        image_path
    )

    gt_df = pd.read_csv(
        gt_path
    )


    # Keep only UID and feature vectors
    text_df = text_df[
        ["uid", "feature_vector"]
    ].rename(
        columns={
            "feature_vector": "text_vector"
        }
    )

    image_df = image_df[
        ["uid", "feature_vector"]
    ].rename(
        columns={
            "feature_vector": "image_vector"
        }
    )


    # Convert stored vectors back to NumPy arrays
    text_df["text_vector"] = (
        text_df["text_vector"]
        .apply(parse_vector)
    )

    image_df["image_vector"] = (
        image_df["image_vector"]
        .apply(parse_vector)
    )


    # Match text and image embeddings by UID
    embeddings_df = text_df.merge(
        image_df,
        on="uid",
        how="inner",
        validate="one_to_one"
    )

    assert len(embeddings_df) == len(text_df)
    assert len(embeddings_df) == len(image_df)


    # Concatenate text and image embeddings
    embeddings_df["feature_vector"] = (
        embeddings_df.apply(
            lambda row: np.concatenate(
                [
                    row["text_vector"],
                    row["image_vector"]
                ]
            ).tolist(),
            axis=1
        )
    )


    # Attach binary labels
    combined_df = embeddings_df[
        ["uid", "feature_vector"]
    ].merge(
        gt_df[
            ["uid", "binary_label"]
        ],
        on="uid",
        how="inner",
        validate="one_to_one"
    )

    assert len(combined_df) == len(embeddings_df)

    # Verify the final multimodal data
    assert combined_df[
        "binary_label"
    ].isin([0, 1]).all()

    assert len(
        combined_df[
            "feature_vector"
        ].iloc[0]
    ) == 1536


    # Store feature vectors as JSON strings
    combined_df["feature_vector"] = (
        combined_df["feature_vector"]
        .apply(json.dumps)
    )


    combined_df.to_csv(
        output_path,
        index=False
    )


    print(
        f"{split_name} combined samples:",
        len(combined_df)
    )

    print(
        f"{split_name} feature dimension:",
        1536
    )

    print(
        f"Saved:",
        output_path
    )

In [ ]:
# Defines the image, text, and output file paths for all three data splits.

TRAIN_TEXT_PATH = os.path.join(
    TEXT_EMBEDDING_FOLDER,
    "train_text_embeddings.csv"
)

VAL_TEXT_PATH = os.path.join(
    TEXT_EMBEDDING_FOLDER,
    "val_text_embeddings.csv"
)

TEST_TEXT_PATH = os.path.join(
    TEXT_EMBEDDING_FOLDER,
    "test_text_embeddings.csv"
)


TRAIN_IMAGE_PATH = os.path.join(
    IMAGE_EMBEDDING_FOLDER,
    "train_image_embeddings_vectors.csv"
)

VAL_IMAGE_PATH = os.path.join(
    IMAGE_EMBEDDING_FOLDER,
    "val_image_embeddings_vectors.csv"
)

TEST_IMAGE_PATH = os.path.join(
    IMAGE_EMBEDDING_FOLDER,
    "test_image_embeddings_vectors.csv"
)


TRAIN_COMBINED_PATH = os.path.join(
    COMBINED_OUTPUT_FOLDER,
    "train_combined_binary.csv"
)

VAL_COMBINED_PATH = os.path.join(
    COMBINED_OUTPUT_FOLDER,
    "val_combined_binary.csv"
)

TEST_COMBINED_PATH = os.path.join(
    COMBINED_OUTPUT_FOLDER,
    "test_combined_binary.csv"
)

In [ ]:
# Combines text and image embeddings for train, validation, and test splits,
# attaches binary labels, and saves the final 1536-dimensional datasets.

combine_embeddings(
    text_path=TRAIN_TEXT_PATH,
    image_path=TRAIN_IMAGE_PATH,
    gt_path=GT_PATH,
    output_path=TRAIN_COMBINED_PATH,
    split_name="Training"
)

combine_embeddings(
    text_path=VAL_TEXT_PATH,
    image_path=VAL_IMAGE_PATH,
    gt_path=GT_PATH,
    output_path=VAL_COMBINED_PATH,
    split_name="Validation"
)

combine_embeddings(
    text_path=TEST_TEXT_PATH,
    image_path=TEST_IMAGE_PATH,
    gt_path=GT_PATH,
    output_path=TEST_COMBINED_PATH,
    split_name="Test"
)

Training combined samples: 2239
Training feature dimension: 1536
Saved: /content/drive/MyDrive/Dataset/train_combined_binary.csv
Validation combined samples: 480
Validation feature dimension: 1536
Saved: /content/drive/MyDrive/Dataset/val_combined_binary.csv
Test combined samples: 480
Test feature dimension: 1536
Saved: /content/drive/MyDrive/Dataset/test_combined_binary.csv


# Train model + validation

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

TRAIN_PATH = "/content/drive/MyDrive/Dataset/train_combined_binary.csv"
VAL_PATH = "/content/drive/MyDrive/Dataset/val_combined_binary.csv"
MODEL_FOLDER = "/content/drive/MyDrive/Dataset/trained_model_binary"

INPUT_DIM = 1536
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
SEED = 42

os.makedirs(MODEL_FOLDER, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))


def parse_vector(value):
    if isinstance(value, str):
        value = json.loads(value)
    return np.asarray(value, dtype=np.float32)


class MultimodalDataset(Dataset):
    def __init__(self, dataframe):
        self.features = np.stack(
            dataframe["feature_vector"].apply(parse_vector).values
        )
        self.labels = (
            dataframe["binary_label"]
            .astype(np.float32)
            .values
            .reshape(-1, 1)
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return (
            torch.tensor(self.features[index], dtype=torch.float32),
            torch.tensor(self.labels[index], dtype=torch.float32)
        )


train_dataset = MultimodalDataset(train_df)
val_dataset = MultimodalDataset(val_df)

assert train_dataset.features.shape[1] == INPUT_DIM
assert val_dataset.features.shape[1] == INPUT_DIM
assert train_dataset.labels.shape[1] == 1
assert val_dataset.labels.shape[1] == 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


class MultimodalClassifier(nn.Module):
    def __init__(self, input_dim=1536, dropout=0.3):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.network(x)


model = MultimodalClassifier(
    input_dim=INPUT_DIM,
    dropout=DROPOUT
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

history = []
best_val_loss = float("inf")
best_epoch = 0

print("\nStarting binary training...\n")

for epoch in range(NUM_EPOCHS):

    model.train()
    total_train_loss = 0.0

    for features, labels in train_loader:
        features = features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(features)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * features.size(0)

    avg_train_loss = total_train_loss / len(train_dataset)

    model.eval()
    total_val_loss = 0.0
    all_val_labels = []
    all_val_probs = []

    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(features)
            loss = criterion(logits, labels)

            total_val_loss += loss.item() * features.size(0)

            probabilities = torch.sigmoid(logits)

            all_val_probs.append(
                probabilities.cpu().numpy()
            )
            all_val_labels.append(
                labels.cpu().numpy()
            )

    avg_val_loss = total_val_loss / len(val_dataset)

    y_true = np.concatenate(all_val_labels).ravel()
    y_prob = np.concatenate(all_val_probs).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    val_accuracy = accuracy_score(y_true, y_pred)
    val_precision = precision_score(
        y_true, y_pred, zero_division=0
    )
    val_recall = recall_score(
        y_true, y_pred, zero_division=0
    )
    val_f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    try:
        val_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        val_auc = np.nan

    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_accuracy": val_accuracy,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1": val_f1,
        "val_auc": val_auc
    })

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Accuracy: {val_accuracy:.4f} | "
        f"Precision: {val_precision:.4f} | "
        f"Recall: {val_recall:.4f} | "
        f"F1: {val_f1:.4f} | "
        f"AUC: {val_auc:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            os.path.join(
                MODEL_FOLDER,
                "best_binary_model.pth"
            )
        )


history_df = pd.DataFrame(history)

history_df.to_csv(
    os.path.join(
        MODEL_FOLDER,
        "binary_training_history.csv"
    ),
    index=False
)

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Using device: cpu


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Dataset/train_combined_binary.csv'

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

BEST_MODEL_PATH = os.path.join(
    MODEL_FOLDER,
    "best_binary_model.pth"
)

model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=device)
)

model.eval()

val_labels = []
val_probs = []

with torch.no_grad():
    for features, labels in val_loader:
        features = features.to(device)

        logits = model(features)
        probabilities = torch.sigmoid(logits)

        val_probs.append(
            probabilities.cpu().numpy()
        )

        val_labels.append(
            labels.numpy()
        )

val_labels = np.concatenate(val_labels).ravel()
val_probs = np.concatenate(val_probs).ravel()

assert len(val_labels) == len(val_dataset)
assert len(val_probs) == len(val_dataset)
assert np.isfinite(val_probs).all()
assert ((val_probs >= 0) & (val_probs <= 1)).all()

val_predictions = (val_probs >= 0.5).astype(int)

accuracy = accuracy_score(
    val_labels,
    val_predictions
)

precision = precision_score(
    val_labels,
    val_predictions,
    zero_division=0
)

recall = recall_score(
    val_labels,
    val_predictions,
    zero_division=0
)

f1 = f1_score(
    val_labels,
    val_predictions,
    zero_division=0
)

auc = roc_auc_score(
    val_labels,
    val_probs
)

cm = confusion_matrix(
    val_labels,
    val_predictions,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)

print("Best model validation results")
print("-----------------------------")
print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1         : {f1:.4f}")
print(f"AUROC      : {auc:.4f}")

print("\nConfusion matrix:")
print(cm)

#Threshold Optimization

In [ ]:
import numpy as np
import pandas as pd

thresholds = np.arange(0.05, 0.96, 0.01)

results = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(int)

    cm = confusion_matrix(
    val_labels,
    predictions,
    labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            val_labels,
            predictions
        ),
        "precision": precision_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            val_labels,
            predictions,
            zero_division=0
        ),
        "specificity": specificity,
        "f1": f1_score(
            val_labels,
            predictions,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(results)

best_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = best_row["threshold"]

print("Best validation threshold")
print("-------------------------")
print(f"Threshold  : {best_threshold:.2f}")
print(f"Accuracy   : {best_row['accuracy']:.4f}")
print(f"Precision  : {best_row['precision']:.4f}")
print(f"Recall     : {best_row['recall']:.4f}")
print(f"Specificity: {best_row['specificity']:.4f}")
print(f"F1         : {best_row['f1']:.4f}")

In [ ]:
near_best = threshold_results[
    threshold_results["threshold"].between(
        best_threshold - 0.05,
        best_threshold + 0.05
    )
]

print(
    near_best.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
baseline_row = threshold_results.iloc[
    (threshold_results["threshold"] - 0.50)
    .abs()
    .argmin()
]

comparison = pd.DataFrame({
    "Metric": [
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1"
        "AUROC"
    ],
    "Default 0.50": [
        baseline_row["threshold"],
        baseline_row["accuracy"],
        baseline_row["precision"],
        baseline_row["recall"],
        baseline_row["specificity"],
        baseline_row["f1"]
        auc
    ],
    "Optimized": [
        best_row["threshold"],
        best_row["accuracy"],
        best_row["precision"],
        best_row["recall"],
        best_row["specificity"],
        best_row["f1"]
    ]
})

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
# Saves the threshold selected on the validation set for use during
# final test evaluation and modality-ablation analysis.

THRESHOLD_PATH = os.path.join(
    MODEL_FOLDER,
    "binary_threshold.json"
)

threshold_info = {
    "selection_set": "validation",
    "selection_metric": "F1",
    "threshold": float(best_threshold)
}

with open(
    THRESHOLD_PATH,
    "w"
) as f:
    json.dump(
        threshold_info,
        f,
        indent=4
    )

print(
    "Saved validation-selected threshold:",
    f"{best_threshold:.2f}"
)

# Test Evaluation

In [ ]:
import json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

TEST_PATH = "/content/drive/MyDrive/Dataset/test_combined_binary.csv"
MODEL_PATH = (
    "/content/drive/MyDrive/Dataset/"
    "trained_model_binary/best_binary_model.pth"
)

BATCH_SIZE = 32
INPUT_DIM = 1536

test_df = pd.read_csv(TEST_PATH)


def parse_vector(value):
    if isinstance(value, str):
        value = json.loads(value)
    return np.asarray(value, dtype=np.float32)


class MultimodalDataset(Dataset):
    def __init__(self, dataframe):
        self.features = np.stack(
            dataframe["feature_vector"]
            .apply(parse_vector)
            .values
        )

        self.labels = (
            dataframe["binary_label"]
            .astype(np.float32)
            .values
            .reshape(-1, 1)
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return (
            torch.tensor(
                self.features[index],
                dtype=torch.float32
            ),
            torch.tensor(
                self.labels[index],
                dtype=torch.float32
            )
        )


test_dataset = MultimodalDataset(test_df)

assert test_dataset.features.shape[1] == INPUT_DIM

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.eval()

test_labels = []
test_probs = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(
            device,
            non_blocking=True
        )

        logits = model(features)
        probabilities = torch.sigmoid(logits)

        test_probs.append(
            probabilities.cpu().numpy()
        )

        test_labels.append(
            labels.numpy()
        )

test_labels = np.concatenate(test_labels).ravel()
test_probs = np.concatenate(test_probs).ravel()

test_predictions = (
    test_probs >= 0.5
).astype(int)

accuracy = accuracy_score(
    test_labels,
    test_predictions
)

precision = precision_score(
    test_labels,
    test_predictions,
    zero_division=0
)

recall = recall_score(
    test_labels,
    test_predictions,
    zero_division=0
)

f1 = f1_score(
    test_labels,
    test_predictions,
    zero_division=0
)

auc = roc_auc_score(
    test_labels,
    test_probs
)

cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)

print("Binary test results")
print("-------------------")

print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1         : {f1:.4f}")
print(f"AUC        : {auc:.4f}")

print("\nConfusion matrix:")
print(cm)

In [ ]:
# Loads the threshold selected on the validation set and evaluates
# test performance using the same threshold.

THRESHOLD_PATH = (
    "/content/drive/MyDrive/Dataset/"
    "trained_model_binary/binary_threshold.json"
)

with open(
    THRESHOLD_PATH,
    "r"
) as f:
    threshold_info = json.load(f)

optimized_threshold = float(
    threshold_info["threshold"]
)

test_predictions_opt = (
    test_probs >= optimized_threshold
).astype(int)

accuracy_opt = accuracy_score(
    test_labels,
    test_predictions_opt
)

precision_opt = precision_score(
    test_labels,
    test_predictions_opt,
    zero_division=0
)

recall_opt = recall_score(
    test_labels,
    test_predictions_opt,
    zero_division=0
)

f1_opt = f1_score(
    test_labels,
    test_predictions_opt,
    zero_division=0
)

cm_opt = confusion_matrix(
    test_labels,
    test_predictions_opt,
    labels=[0, 1]
)

tn, fp, fn, tp = cm_opt.ravel()

specificity_opt = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)

print("\nValidation-optimized threshold results")
print("--------------------------------------")

print(f"Threshold  : {optimized_threshold:.2f}")
print(f"Accuracy   : {accuracy_opt:.4f}")
print(f"Precision  : {precision_opt:.4f}")
print(f"Recall     : {recall_opt:.4f}")
print(f"Specificity: {specificity_opt:.4f}")
print(f"F1         : {f1_opt:.4f}")
print(f"AUC        : {auc:.4f}")

print("\nConfusion matrix:")
print(cm_opt)

In [ ]:
# Compares test performance at the default and validation-selected thresholds.

comparison = pd.DataFrame({

    "Metric": [
        "Threshold",
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1",
        "AUROC"
    ],

    "Default 0.50": [
        0.50,
        accuracy,
        precision,
        recall,
        specificity,
        f1,
        auc
    ],

    "Optimized": [
        optimized_threshold,
        accuracy_opt,
        precision_opt,
        recall_opt,
        specificity_opt,
        f1_opt,
        auc
    ]
})

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# Approach 2

#Load libraries, paths, device, and model

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

TEST_PATH = "/content/drive/MyDrive/Dataset/test_combined_binary.csv"

MODEL_PATH = (
    "/content/drive/MyDrive/Dataset/"
    "trained_model_binary/best_binary_model.pth"
)

THRESHOLD_PATH = (
    "/content/drive/MyDrive/Dataset/"
    "trained_model_binary/binary_threshold.json"
)

with open(
    THRESHOLD_PATH,
    "r"
) as f:
    threshold_info = json.load(f)

THRESHOLD = float(
    threshold_info["threshold"]
)

print(
    "Using validation-selected threshold:",
    THRESHOLD
)

TEXT_DIM = 768
IMAGE_DIM = 768
INPUT_DIM = 1536

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


class MultimodalClassifier(nn.Module):
    def __init__(self, input_dim=1536, dropout=0.3):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.network(x)


model = MultimodalClassifier(
    input_dim=INPUT_DIM,
    dropout=0.3
).to(device)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model.eval()

print("Model loaded successfully.")

#Load test data and create ablated inputs

In [ ]:
def parse_vector(value):
    if isinstance(value, str):
        value = json.loads(value)
    return np.asarray(value, dtype=np.float32)


test_df = pd.read_csv(TEST_PATH)

X_test = np.stack(
    test_df["feature_vector"]
    .apply(parse_vector)
    .values
)

y_test = (
    test_df["binary_label"]
    .astype(int)
    .values
)

assert X_test.shape[1] == INPUT_DIM

X_full = X_test.copy()

# Keep text, remove image
X_text_only = X_test.copy()
X_text_only[:, TEXT_DIM:] = 0

# Keep image, remove text
X_image_only = X_test.copy()
X_image_only[:, :TEXT_DIM] = 0

print("Test samples:", len(X_test))
print("Feature dimension:", X_test.shape[1])

#Get probabilities for each modality condition

In [ ]:
def get_probabilities(X, batch_size=32):
    probabilities = []

    with torch.no_grad():
        for start in range(0, len(X), batch_size):

            batch = torch.tensor(
                X[start:start + batch_size],
                dtype=torch.float32
            ).to(device)

            logits = model(batch)

            probs = torch.sigmoid(logits)

            probabilities.append(
                probs.cpu().numpy()
            )

    return np.concatenate(
        probabilities
    ).ravel()


prob_full = get_probabilities(X_full)

prob_text_only = get_probabilities(
    X_text_only
)

prob_image_only = get_probabilities(
    X_image_only
)

print("Probabilities generated.")

#Evaluate all three conditions

In [ ]:
# Evaluates each modality condition using the threshold selected
# previously on the validation set.

def evaluate_condition(
    y_true,
    y_prob,
    threshold
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Specificity": specificity,

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "AUROC": roc_auc_score(
            y_true,
            y_prob
        )
    }

In [ ]:
# Evaluates the full multimodal model and both modality-ablation conditions.

results = {
    "Image + Text": evaluate_condition(
        y_test,
        prob_full,
        THRESHOLD
    ),

    "Text only (Image ablated)": evaluate_condition(
        y_test,
        prob_text_only,
        THRESHOLD
    ),

    "Image only (Text ablated)": evaluate_condition(
        y_test,
        prob_image_only,
        THRESHOLD
    )
}

results_df = pd.DataFrame(
    results
).T

print(
    results_df.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)

#Compute modality performance drops

In [ ]:
auc_full = results_df.loc[
    "Image + Text",
    "AUROC"
]

auc_text_only = results_df.loc[
    "Text only (Image ablated)",
    "AUROC"
]

auc_image_only = results_df.loc[
    "Image only (Text ablated)",
    "AUROC"
]

f1_full = results_df.loc[
    "Image + Text",
    "F1"
]

f1_text_only = results_df.loc[
    "Text only (Image ablated)",
    "F1"
]

f1_image_only = results_df.loc[
    "Image only (Text ablated)",
    "F1"
]


image_drop_auc = (
    auc_full - auc_text_only
)

text_drop_auc = (
    auc_full - auc_image_only
)

image_drop_f1 = (
    f1_full - f1_text_only
)

text_drop_f1 = (
    f1_full - f1_image_only
)


print("AUROC performance drop")
print("----------------------")
print(
    f"Image contribution drop: "
    f"{image_drop_auc:.4f}"
)

print(
    f"Text contribution drop : "
    f"{text_drop_auc:.4f}"
)

print("\nF1 performance drop")
print("-------------------")
print(
    f"Image contribution drop: "
    f"{image_drop_f1:.4f}"
)

print(
    f"Text contribution drop : "
    f"{text_drop_f1:.4f}"
)

#Compute relative modality contribution

In [ ]:
total_auc_drop = (
    image_drop_auc +
    text_drop_auc
)

if (
    image_drop_auc > 0 and
    text_drop_auc > 0
):

    image_contribution_auc = (
        image_drop_auc /
        total_auc_drop
    ) * 100

    text_contribution_auc = (
        text_drop_auc /
        total_auc_drop
    ) * 100

    print(
        "Relative modality contribution "
        "based on AUROC"
    )

    print("-------------------------------")

    print(
        f"Image contribution: "
        f"{image_contribution_auc:.2f}%"
    )

    print(
        f"Text contribution : "
        f"{text_contribution_auc:.2f}%"
    )

else:
    print(
        "At least one AUROC drop is "
        "non-positive."
    )

    print(
        "Do not interpret normalized "
        "contribution percentages directly."
    )

#Final summary table

In [ ]:
summary_df = pd.DataFrame({
    "Modality": [
        "Image",
        "Text"
    ],

    "AUROC Drop": [
        image_drop_auc,
        text_drop_auc
    ],

    "F1 Drop": [
        image_drop_f1,
        text_drop_f1
    ]
})

if (
    image_drop_auc > 0 and
    text_drop_auc > 0
):

    summary_df[
        "Relative AUROC Contribution (%)"
    ] = [
        image_contribution_auc,
        text_contribution_auc
    ]

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)